In [4]:
import os, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_sobel_gradients(image):
    """Sobel edge detection. Uses image.device"""
    dev = image.device
    kx  = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], device=dev).view(1,1,3,3).float()
    ky  = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], device=dev).view(1,1,3,3).float()
    return F.conv2d(image, kx, padding=1), F.conv2d(image, ky, padding=1)


class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, channels):
        super().__init__()
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1),
            nn.ReLU(),
            nn.Conv2d(channels // 8, channels, 1),
            nn.Sigmoid()
        )
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x * self.ca(x)
        s = torch.cat([torch.mean(x, 1, keepdim=True),
                       torch.max(x,  1, keepdim=True)[0]], dim=1)
        return x * self.sa(s)

class PreActResidualUnit(nn.Module):
    """
    Implements a Full Pre-activation Residual Unit: BN -> ReLU -> Conv
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super(PreActResidualUnit, self).__init__()

        # Branch 1: Pre-activated Residual Function
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)

        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)

        # Branch 2: Shortcut Connection (Identity Mapping)
        # If dimensions change (stride > 1 or channel mismatch), apply a 1x1 conv to match.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # Apply pre-activation
        pre_act = self.relu(self.bn1(x))

        # Residual signal
        out = self.conv1(pre_act)
        out = self.conv2(self.relu(self.bn2(out)))

        # Combine with identity shortcut
        return out + self.shortcut(x)

class LAR_UNet(nn.Module):
    """
    Enhanced LAR-U-Net integrating Deep Residual Learning and CBAM.
    Replaces Max-Pooling with Strided Convolutions for better feature retention.
    """
    def __init__(self, in_nc=1, out_nc=1, features=64):
        super(LAR_UNet, self).__init__()

        # Initial Gradient Augmentation (from LAR-U-Net original design)
        # Input is 3 channels: Grayscale + Gx + Gy
        self.init_conv = nn.Conv2d(3, features, kernel_size=3, padding=1)

        # Encoder Path
        self.enc_block1 = PreActResidualUnit(features, features)
        self.att1 = CBAM(features)

        # Downsampling via stride=2 convolution in the residual unit
        self.enc_block2 = PreActResidualUnit(features, features * 2, stride=2)
        self.att2 = CBAM(features * 2)

        # Bridge (Bottleneck)
        self.bridge = PreActResidualUnit(features * 2, features * 4, stride=2)

        # Decoder Path
        self.up1 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.dec_block1 = PreActResidualUnit(features * 4, features * 2) # Concat + Residual

        self.up2 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.dec_block2 = PreActResidualUnit(features * 2, features) # Concat + Residual

        # Final Noise Prediction (Residual Learning Strategy)
        self.final = nn.Conv2d(features, out_nc, kernel_size=1)

    def forward(self, y):
        # 1. Gradient-Augmented Input Processing
        gx, gy = get_sobel_gradients(y)
        x = torch.cat([y, gx, gy], dim=1)
        x = self.init_conv(x)

        # 2. Encoding Path (with Skip Connections)
        e1 = self.att1(self.enc_block1(x))
        e2 = self.att2(self.enc_block2(e1))

        # 3. Bridge
        b = self.bridge(e2)

        # 4. Decoding Path (Concatenation)
        d1 = self.up1(b)
        d1 = self.dec_block1(torch.cat([d1, e2], dim=1))

        d2 = self.up2(d1)
        d2 = self.dec_block2(torch.cat([d2, e1], dim=1))

        # 5. Output: Subtract predicted noise from noisy input
        noise_map = self.final(d2)
        return y - noise_map



In [6]:
from google.colab import files
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 1. Initialize the architecture
# (Ensure the DeepRes_LAR_UNet class is defined in a previous cell)
model = LAR_UNet().to(device)

# 2. Upload the .pth file from your local computer
print("Please upload the 'lar_unet_weights.pth' file:")
uploaded = files.upload()

# 3. Load the weights into the model
weights_path = 'lar_unet_weights.pth'
if weights_path in uploaded:
    # Load state dict with map_location to ensure it works on both CPU and GPU
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"\nSuccessfully loaded model weights from {weights_path}")
else:
    print(f"\nError: File {weights_path} not found in upload.")

PyTorch version : 2.10.0+cu128
CUDA available  : True
Please upload the 'lar_unet_weights.pth' file:


Saving lar_unet_weights.pth to lar_unet_weights (1).pth

Error: File lar_unet_weights.pth not found in upload.


In [7]:
class BSDS500Denoise(Dataset):
    def __init__(self, root_dir, sigma=25):
        self.root_dir    = root_dir
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith(('.jpg', '.png'))]
        self.sigma       = sigma / 255.0
        self.transform   = transforms.Compose([
            transforms.CenterCrop(128),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.root_dir, self.image_files[idx])
        clean_img = self.transform(Image.open(img_path).convert('L'))
        noise     = torch.randn(clean_img.size()) * self.sigma
        noisy_img = (clean_img + noise).clamp(0, 1)
        return noisy_img, clean_img


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("balraj98/berkeley-segmentation-dataset-500-bsds500")

print("Path to dataset files:", path)

100%|██████████| 56.0M/56.0M [00:00<00:00, 173MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/versions/1


In [9]:
# Paths — confirmed correct structure
base       = "/root/.cache/kagglehub/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/versions/1"
train_path = os.path.join(base, "images", "train")
val_path   = os.path.join(base, "images", "val")
test_path  = os.path.join(base, "images", "test")
SIGMA=25
BATCH_SIZE=8

loader_train = DataLoader(BSDS500Denoise(train_path, sigma=SIGMA), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
loader_val   = DataLoader(BSDS500Denoise(val_path,   sigma=SIGMA), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
loader_test  = DataLoader(BSDS500Denoise(test_path,  sigma=SIGMA), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [10]:
import numpy as np
import torch

# 1. Prepare for Evaluation
model.eval()
model.to(device)
all_psnr = []

print(f"Starting evaluation on {len(loader_test.dataset)} BSDS500 test images...")

# 2. Inference Loop
with torch.no_grad():
    for noisy, clean in loader_test:
        # Move batch to device
        noisy, clean = noisy.to(device), clean.to(device)

        # Predict noise and recover image: x_hat = y - R(y)
        denoised = model(noisy)

        # Convert tensors to NumPy arrays (B, C, H, W)
        # We process individual images in the batch to get precise PSNR
        clean_np = clean.cpu().numpy()
        denoised_np = denoised.cpu().numpy()

        for i in range(clean_np.shape[0]):
            # Calculate MSE for the single image
            mse = np.mean((clean_np[i] - denoised_np[i]) ** 2)

            # Calculate PSNR: 10 * log10(MAX^2 / MSE)
            # MAX is 1.0 since our images are normalized [0, 1]
            if mse > 0:
                psnr = 10 * np.log10(1.0 / mse)
            else:
                psnr = 100.0  # Represents a perfect match

            all_psnr.append(psnr)

# 3. Final Metrics Summary
average_psnr = np.mean(all_psnr)
std_psnr = np.std(all_psnr)

print(f"\n" + "="*30)
print(f" BSDS500 TEST SET RESULTS")
print(f"="*30)
print(f"Total Images   : {len(all_psnr)}")
print(f"Average PSNR   : {average_psnr:.4f} dB")
print(f"Std Deviation  : {std_psnr:.4f} dB")
print(f"="*30)

Starting evaluation on 200 BSDS500 test images...

 BSDS500 TEST SET RESULTS
Total Images   : 200
Average PSNR   : 19.9205 dB
Std Deviation  : 0.1890 dB
